# Differential usage: transcript isoforms and ORFs

One objective: for genes whose isoform mix may differ between the groups, show the share each isoform takes in every sample. The tests are the DRIMSeq tables from S4. Each panel is drawn with all samples and again without the samples named in `exclude`.

In [ ]:
source("src/heatmap.R")   # read_groups
source("src/usage.R")
source("src/paths.R")
options(repr.plot.width = 11)
loc <- run_locations()        # every folder of the run, from paths.json
comparison <- loc$comparison
exclude <- c("420_20190402")   # samples to leave out in the second drawing; character(0) skips it
n_genes <- 4                    # how many genes to draw in each panel
groups <- read_groups(file.path(loc$leafcutter, paste0(comparison, ".lr_leafcutter.groups_file.txt")))
g2 <- if (length(exclude) > 0) drop_samples(groups, exclude) else groups
label <- paste(exclude, collapse = ", ")

**Explanation.** The line that starts with `loc` reads `paths.json`, the one place where the run's results folder, the comparison name and the dataset name are set. The other cells use short names such as `loc$du_transcript`, so to look at another run you change `paths.json` and nothing in the notebook. DRIMSeq asks a different question from the volcano plots: not whether a gene is expressed more, but whether it spends its expression on different isoforms. `exclude` names the samples to leave out in the second panels; `character(0)` skips them. `n_genes` is how many genes each panel shows.

In [ ]:
dtu <- prepare_usage(read_usage(file.path(loc$du_transcript, paste0(comparison, "_DTU_transcript_DRIMSeq_summary.txt"))), "transcript")
usage_summary(dtu)

**Explanation.** How many genes DRIMSeq tested for differences in isoform use between the groups, how many have an adjusted p value below 0.05 and 0.1, and the smallest p values. The adjusted p value is the raw p value corrected for testing many genes (Benjamini-Hochberg), so it is the one to judge by.

In [ ]:
pvalue_histogram(dtu, "Transcript isoforms: raw p value of every tested gene")

**Explanation.** One bar per range of raw p values, counting genes. A roughly flat histogram means the groups do not differ in isoform use at this sample size. A pile of genes at the far left, near 0, would mean real differences. When many genes share the same adjusted p value, that is the correction at work on p values that are all large: nothing stands out from chance.

In [ ]:
head(gene_usage_table(dtu), 20)

**Explanation.** The 20 genes with the smallest p values, with their raw `p` and adjusted `adj_p`. `NA` means DRIMSeq did not test that gene (too little expression, or fewer isoforms than the filters need). Use `n` in `head()` to show more.

In [ ]:
options(repr.plot.height = 3.3 * n_genes)
usage_panel(dtu, groups, n = n_genes)
if (length(exclude) > 0) {
  cat("without", label, "\n")
  print(usage_panel(dtu, g2, n = n_genes))
}

**Explanation.** The genes with the smallest p values, one above the other. These are the smallest p values, not necessarily significant: the adjusted p is in each title. Each bar is a sample, split by group; the colours are the gene's isoforms and the bar height is the share of the gene's expression that isoform takes. If the colour pattern differs between the groups, the gene switches isoforms. A sample with no expression of the gene is left blank. Bars stop short of 1.0 because DRIMSeq drops very rare isoforms before testing; the missing part is those isoforms. The second set leaves out the samples in `exclude`; the p values are the pipeline's, computed with all samples.

In [ ]:
du_orf <- prepare_usage(read_usage(file.path(loc$du_orf, paste0(comparison, "_DU_ORF_DRIMSeq_summary.txt"))), "orf")
usage_summary(du_orf)

**Explanation.** The same for open reading frames (ORFs): whether the predicted protein-coding versions of a gene are used differently between the groups. The same counts as above, for ORFs.

In [ ]:
pvalue_histogram(du_orf, "ORFs: raw p value of every tested gene")

**Explanation.** The same p value histogram for ORF usage.

In [ ]:
head(gene_usage_table(du_orf), 20)

**Explanation.** The 20 genes with the smallest p values for ORF usage, raw and adjusted.

In [ ]:
options(repr.plot.height = 3.3 * n_genes)
usage_panel(du_orf, groups, n = n_genes)
if (length(exclude) > 0) {
  cat("without", label, "\n")
  print(usage_panel(du_orf, g2, n = n_genes))
}

**Explanation.** The ORF version of the panels: the predicted protein-coding versions of a gene, drawn the same way, with and without the samples in `exclude`.